# Nighttime Yard Traffic Forecasting Using Bi-LSTM — Public Demo

This notebook is a **public portfolio demo** for a yard traffic forecasting project.  
It uses **synthetic data only** and does not contain original operational data, internal company files, or confidential outputs.

## Demo Objectives

- Generate synthetic delivery and discharge traffic data.
- Apply a simple Kalman-style smoothing approach for missing values.
- Create time-series sequences for forecasting.
- Train Bi-LSTM models for delivery and discharge forecasting.
- Evaluate model performance using MAE, RMSE, MAPE, and Index of Agreement.
- Generate a 30-day demo forecast.


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)


## 2. Load or Generate Synthetic Dataset

The original operational dataset is not included due to confidentiality.  
This section loads the synthetic sample data included in the repository. If the file is not available, the notebook will generate synthetic data automatically.


In [ ]:
from pathlib import Path

data_path = Path("../data/synthetic_yard_traffic_sample.csv")

if data_path.exists():
    df = pd.read_csv(data_path, parse_dates=["date"])
else:
    rng = np.random.default_rng(42)
    dates = pd.date_range("2022-01-01", "2024-12-31", freq="D")
    n = len(dates)
    day = np.arange(n)

    weekly = np.sin(2 * np.pi * day / 7)
    monthly = np.sin(2 * np.pi * day / 30)
    trend = np.linspace(0, 60, n)

    delivery = 360 + 180 * weekly + 60 * monthly + trend + rng.normal(0, 75, n)
    discharge = 470 + 70 * np.sin(2 * np.pi * day / 14) + 25 * monthly + rng.normal(0, 140, n)

    spike_idx = rng.choice(n, size=35, replace=False)
    delivery[spike_idx] += rng.uniform(120, 280, size=len(spike_idx))
    discharge[spike_idx] += rng.uniform(150, 380, size=len(spike_idx))

    df = pd.DataFrame({
        "date": dates,
        "delivery_volume": np.clip(delivery, 20, None),
        "discharge_volume": np.clip(discharge, 20, None)
    })

    for col in ["delivery_volume", "discharge_volume"]:
        miss_idx = rng.choice(n, size=20, replace=False)
        df.loc[miss_idx, col] = np.nan

df.head()


## 3. Exploratory Visualization

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["delivery_volume"], label="Delivery Volume")
plt.plot(df["date"], df["discharge_volume"], label="Discharge Volume")
plt.title("Synthetic Yard Traffic Data")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.legend()
plt.grid(True)
plt.show()

print("Missing values:")
print(df.isna().sum())


## 4. Missing Value Handling with Simple Kalman-Style Smoothing

This is a simplified smoothing function for public demo purposes.  
The original internal workflow and operational dataset are not included.


In [ ]:
def simple_kalman_smoothing(series, process_variance=1e-4, measurement_variance=0.05):
    """Simple 1D Kalman-style smoothing for demonstration purposes."""
    values = pd.Series(series).astype(float).copy()
    values = values.interpolate(method="linear").bfill().ffill()

    xhat = np.zeros(len(values))
    p = np.zeros(len(values))
    xhat[0] = values.iloc[0]
    p[0] = 1.0

    for k in range(1, len(values)):
        # Prediction
        xhat_minus = xhat[k - 1]
        p_minus = p[k - 1] + process_variance

        # Update
        kalman_gain = p_minus / (p_minus + measurement_variance)
        xhat[k] = xhat_minus + kalman_gain * (values.iloc[k] - xhat_minus)
        p[k] = (1 - kalman_gain) * p_minus

    return xhat

df_processed = df.copy()
df_processed["delivery_smoothed"] = simple_kalman_smoothing(df_processed["delivery_volume"])
df_processed["discharge_smoothed"] = simple_kalman_smoothing(df_processed["discharge_volume"])

df_processed.head()


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df_processed["date"], df_processed["delivery_smoothed"], label="Delivery Smoothed")
plt.plot(df_processed["date"], df_processed["discharge_smoothed"], label="Discharge Smoothed")
plt.title("Smoothed Synthetic Yard Traffic Data")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.legend()
plt.grid(True)
plt.show()


## 5. Feature Engineering

Calendar-based features are added to help the model learn weekly and monthly patterns in the synthetic data.


In [ ]:
df_model = df_processed.copy()

df_model["dayofweek"] = df_model["date"].dt.dayofweek
df_model["month"] = df_model["date"].dt.month

df_model["dow_sin"] = np.sin(2 * np.pi * df_model["dayofweek"] / 7)
df_model["dow_cos"] = np.cos(2 * np.pi * df_model["dayofweek"] / 7)
df_model["month_sin"] = np.sin(2 * np.pi * df_model["month"] / 12)
df_model["month_cos"] = np.cos(2 * np.pi * df_model["month"] / 12)

features = [
    "delivery_smoothed",
    "discharge_smoothed",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]

df_model[features].head()


## 6. Train-Test Split and Sequence Generation

In [ ]:
def create_sequences(data, target_index, lookback=30):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i, :])
        y.append(data[i, target_index])
    return np.array(X), np.array(y)

lookback = 30
train_size = int(len(df_model) * 0.80)

scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(df_model[features])

target_delivery_index = features.index("delivery_smoothed")
target_discharge_index = features.index("discharge_smoothed")

X_delivery, y_delivery = create_sequences(scaled_values, target_delivery_index, lookback)
X_discharge, y_discharge = create_sequences(scaled_values, target_discharge_index, lookback)

train_cut = train_size - lookback

X_delivery_train, X_delivery_test = X_delivery[:train_cut], X_delivery[train_cut:]
y_delivery_train, y_delivery_test = y_delivery[:train_cut], y_delivery[train_cut:]

X_discharge_train, X_discharge_test = X_discharge[:train_cut], X_discharge[train_cut:]
y_discharge_train, y_discharge_test = y_discharge[:train_cut], y_discharge[train_cut:]

X_delivery_train.shape, X_delivery_test.shape


## 7. Build Bi-LSTM Model

In [ ]:
def build_bilstm_model(input_shape):
    model = Sequential([
        Bidirectional(LSTM(32, return_sequences=True), input_shape=input_shape),
        Dropout(0.20),
        Bidirectional(LSTM(16)),
        Dropout(0.20),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )
    return model

input_shape = (X_delivery_train.shape[1], X_delivery_train.shape[2])
delivery_model = build_bilstm_model(input_shape)
discharge_model = build_bilstm_model(input_shape)

delivery_model.summary()


## 8. Train Models

The number of epochs is intentionally small so the notebook can run quickly in Google Colab.


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_delivery = delivery_model.fit(
    X_delivery_train, y_delivery_train,
    validation_split=0.20,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

history_discharge = discharge_model.fit(
    X_discharge_train, y_discharge_train,
    validation_split=0.20,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_delivery.history["loss"], label="Delivery Training Loss")
plt.plot(history_delivery.history["val_loss"], label="Delivery Validation Loss")
plt.title("Delivery Model Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history_discharge.history["loss"], label="Discharge Training Loss")
plt.plot(history_discharge.history["val_loss"], label="Discharge Validation Loss")
plt.title("Discharge Model Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## 9. Evaluation Metrics

In [ ]:
def inverse_single_target(scaled_target, target_index, n_features):
    dummy = np.zeros((len(scaled_target), n_features))
    dummy[:, target_index] = scaled_target.reshape(-1)
    inv = scaler.inverse_transform(dummy)
    return inv[:, target_index]

def index_of_agreement(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    numerator = np.sum((predicted - actual) ** 2)
    denominator = np.sum((np.abs(predicted - np.mean(actual)) + np.abs(actual - np.mean(actual))) ** 2)
    return 1 - (numerator / denominator)

def evaluate_forecast(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / np.maximum(actual, 1e-8))) * 100
    ia = index_of_agreement(actual, predicted)
    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "IA": ia
    }

delivery_pred_scaled = delivery_model.predict(X_delivery_test).reshape(-1)
discharge_pred_scaled = discharge_model.predict(X_discharge_test).reshape(-1)

delivery_actual = inverse_single_target(y_delivery_test, target_delivery_index, len(features))
delivery_pred = inverse_single_target(delivery_pred_scaled, target_delivery_index, len(features))

discharge_actual = inverse_single_target(y_discharge_test, target_discharge_index, len(features))
discharge_pred = inverse_single_target(discharge_pred_scaled, target_discharge_index, len(features))

delivery_metrics = evaluate_forecast(delivery_actual, delivery_pred)
discharge_metrics = evaluate_forecast(discharge_actual, discharge_pred)

pd.DataFrame([delivery_metrics, discharge_metrics], index=["Delivery", "Discharge"])


## 10. Actual vs Predicted Visualization

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(delivery_actual, label="Actual Delivery")
plt.plot(delivery_pred, label="Predicted Delivery")
plt.title("Bi-LSTM Predicted vs Actual Delivery — Synthetic Data")
plt.xlabel("Test Time Index")
plt.ylabel("Volume")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(discharge_actual, label="Actual Discharge")
plt.plot(discharge_pred, label="Predicted Discharge")
plt.title("Bi-LSTM Predicted vs Actual Discharge — Synthetic Data")
plt.xlabel("Test Time Index")
plt.ylabel("Volume")
plt.legend()
plt.grid(True)
plt.show()


## 11. Simple 30-Day Demo Forecast

This section creates a simplified iterative forecast for portfolio demonstration only.


In [ ]:
def iterative_forecast(model, scaled_data, target_index, steps=30, lookback=30):
    current_sequence = scaled_data[-lookback:].copy()
    preds_scaled = []

    for _ in range(steps):
        x_input = current_sequence.reshape(1, lookback, current_sequence.shape[1])
        pred = model.predict(x_input, verbose=0)[0, 0]
        preds_scaled.append(pred)

        # Update next row by copying last row and replacing target value
        next_row = current_sequence[-1].copy()
        next_row[target_index] = pred
        current_sequence = np.vstack([current_sequence[1:], next_row])

    return np.array(preds_scaled)

forecast_steps = 30

delivery_forecast_scaled = iterative_forecast(
    delivery_model, scaled_values, target_delivery_index, forecast_steps, lookback
)

discharge_forecast_scaled = iterative_forecast(
    discharge_model, scaled_values, target_discharge_index, forecast_steps, lookback
)

delivery_forecast = inverse_single_target(delivery_forecast_scaled, target_delivery_index, len(features))
discharge_forecast = inverse_single_target(discharge_forecast_scaled, target_discharge_index, len(features))

forecast_dates = pd.date_range(df_model["date"].max() + pd.Timedelta(days=1), periods=forecast_steps, freq="D")

forecast_df = pd.DataFrame({
    "date": forecast_dates,
    "forecast_delivery": delivery_forecast,
    "forecast_discharge": discharge_forecast
})

forecast_df.head()


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df_model["date"].tail(120), df_model["delivery_smoothed"].tail(120), label="Historical Delivery")
plt.plot(forecast_df["date"], forecast_df["forecast_delivery"], label="30-Day Delivery Forecast")
plt.title("30-Day Delivery Forecast — Synthetic Data")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(df_model["date"].tail(120), df_model["discharge_smoothed"].tail(120), label="Historical Discharge")
plt.plot(forecast_df["date"], forecast_df["forecast_discharge"], label="30-Day Discharge Forecast")
plt.title("30-Day Discharge Forecast — Synthetic Data")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.legend()
plt.grid(True)
plt.show()


## 12. Conclusion

This public demo shows how Bi-LSTM can be used in a time-series forecasting workflow for yard traffic analytics.

Key points:

- The notebook uses synthetic data only.
- The workflow demonstrates preprocessing, smoothing, sequence creation, modeling, evaluation, and forecasting.
- The demo is suitable for GitHub portfolio purposes while protecting confidential operational data.
